# IBKR API notebook for fetching historical market data

#### Prerequisites: 
- ```pip install -r requirements.txt``` to install the necessary packages.

- Launch Trader Workstation (TWS) and enable ActiveX API (```File->Global Configuration->API->Settings``` then check ```Enable ActiveX and Socket Clients``` and uncheck ```Read-Only API```. Do not forget to apply the settings).

#### 1. Connection to IBKR API local gateway

```File->Global Configuration->API->Settings```
et cocher 
```Enable ActiveX and Socket Clients```
et décocher 
```Read-Only API```

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
import pandas as pd
from ib_async import *
util.startLoop()
global historical_data_interval, duration

ib = IB()
ib.connect('127.0.0.1', 7497, clientId=14)
if ib.isConnected():
    print("✅ Connected to IBKR API")
else:
    print("❌Failed to connect to IBKR API")
util.logToConsole(logging.INFO)

2025-08-27 11:24:01,490 - INFO - Connecting to 127.0.0.1:7497 with clientId 14...
2025-08-27 11:24:01,493 - INFO - Connected
2025-08-27 11:24:01,561 - INFO - Logged on to server version 178
2025-08-27 11:24:01,624 - INFO - Warning 2104, reqId -1: La connexion de donn\u00e9es de march\u00e9 est OK:usfuture
2025-08-27 11:24:01,631 - INFO - Warning 2104, reqId -1: La connexion de donn\u00e9es de march\u00e9 est OK:eufarm
2025-08-27 11:24:01,634 - INFO - Warning 2104, reqId -1: La connexion de donn\u00e9es de march\u00e9 est OK:cashfarm
2025-08-27 11:24:01,636 - INFO - Warning 2104, reqId -1: La connexion de donn\u00e9es de march\u00e9 est OK:eufarmnj
2025-08-27 11:24:01,637 - INFO - Warning 2104, reqId -1: La connexion de donn\u00e9es de march\u00e9 est OK:usfarm
2025-08-27 11:24:01,638 - INFO - Warning 2106, reqId -1: La connexion de donn\u00e9es HMDS est OK:euhmds
2025-08-27 11:24:01,638 - INFO - Warning 2106, reqId -1: La connexion de donn\u00e9es HMDS est OK:ushmds
2025-08-27 11:24:01

✅ Connected to IBKR API


2025-08-27 11:24:23,386 - INFO - Warning 2174, reqId 3: Avertissement: You submitted request with date-time attributes without explicit time zone. Please switch to use yyyymmdd-hh:mm:ss in UTC or use instrument time zone, like US/Eastern. Implied time zone functionality will be removed in the next API release, contract: Index(symbol='SPX', exchange='CBOE', currency='USD')
2025-08-27 15:26:08,182 - INFO - Warning 2105, reqId -1: La connexion de donn\u00e9es HMDS est rompue:ushmds
2025-08-27 15:26:08,514 - INFO - Warning 165, reqId 6: Message d'erreur du Service de Donn\u00e9es Historiques de March\u00e9:HMDS server disconnect occurred.  Attempting reconnection..., contract: Index(symbol='SPX', exchange='CBOE', currency='USD')
2025-08-27 15:26:11,169 - INFO - Warning 2106, reqId -1: La connexion de donn\u00e9es HMDS est OK:ushmds
2025-08-27 15:26:12,086 - INFO - Warning 165, reqId 6: Message d'erreur du Service de Donn\u00e9es Historiques de March\u00e9:HMDS server connection was success

## Request Historical data

#### 2. Choose your contract

In [ ]:
# Define the contract HERE
#contract = CFD('IBUST100', 'SMART', 'USD')
#contract = Forex(pair="EURUSD", exchange='IDEALPRO')
#contract = Stock(symbol='AAPL', exchange='SMART', currency='USD')
# contract = Index('NDX', 'NASDAQ', 'USD')
contract = Index('SPX', 'CBOE', 'USD')
#CAC 40
# contract = Index('FCHI', 'EUREX', 'EUR')


# Below => just some printing on contract chosen
contract_details = ib.reqContractDetails(contract)
# Extract and display the desired fields from contract_details
filtered_details = [
    {
        "secType": detail.contract.secType,
        "conId": detail.contract.conId,
        "symbol": detail.contract.symbol,
        "exchange": detail.contract.exchange,
        "longName": detail.longName,
        "timezoneId": detail.timeZoneId,
        "tradingHours": "\n".join(
            [f"  {segment}" for segment in detail.tradingHours.split(";")]
        ),
        "liquidHours": "\n".join(
            [f"  {segment}" for segment in detail.liquidHours.split(";")]
        ),
        "minSize": detail.minSize,
    }
    for detail in contract_details
]

# Print the filtered details in a clear format
for idx, detail in enumerate(filtered_details, start=1):
    print(f"Contract Detail {idx}:")
    for key, value in detail.items():
        print(f"  {key}: {value}")
    print()

#### (Optional) Check first data timestamp available

In [ ]:
timestamp = ib.reqHeadTimeStamp(contract, whatToShow='TRADES', useRTH=False)
formatted_time = timestamp.strftime('%B %d, %Y, %H:%M')
logging.info(f"First date of data available: {formatted_time}")
logging.info(f"Timestamp: {timestamp}")

#### 3. End date choice for data request

In [ ]:
# yesterday's date
end_date = (pd.Timestamp.now(tz='UTC') - pd.DateOffset(days=1)).strftime('%Y%m%d %H:%M:%S')

In [ ]:
# today's date
end_date = pd.Timestamp.now(tz='UTC').strftime('%Y%m%d-%H:%M:%S')

In [ ]:
# custom end date
# format expected : YYYYMMDD HH:MM:SS
end_date = '20221125 22:00:00'

#### 4. Main loop for fetching historical data

Args
- **contract**: Contract of interest.  
- **endDateTime**:  
    - Can be set to `''` to indicate the current time.  
    - Can be given as a `datetime.date` or `datetime.datetime`.  
    - Can be given as a string in `'yyyyMMdd HH:mm:ss'` format.  
    - If no timezone is given, the TWS login timezone is used.  
- **durationStr**: Time span of all the bars. Examples:  
    - `'60 S'`, `'30 D'`, `'13 W'`, `'6 M'`, `'10 Y'`.  
- **barSizeSetting**: Time period of one bar. Must be one of:  
    - `'1 secs'`, `'5 secs'`, `'10 secs'`, `'15 secs'`, `'30 secs'`,  
    - `'1 min'`, `'2 mins'`, `'3 mins'`, `'5 mins'`, `'10 mins'`, `'15 mins'`,  
    - `'20 mins'`, `'30 mins'`,  
    - `'1 hour'`, `'2 hours'`, `'3 hours'`, `'4 hours'`, `'8 hours'`,  
    - `'1 day'`, `'1 week'`, `'1 month'`.  
- **whatToShow**: Specifies the source for constructing bars. Must be one of:  
    - `'TRADES'`, `'MIDPOINT'`, `'BID'`, `'ASK'`, `'BID_ASK'`,  
    - `'ADJUSTED_LAST'`, `'HISTORICAL_VOLATILITY'`, `'OPTION_IMPLIED_VOLATILITY'`,  
    - `'REBATE_RATE'`, `'FEE_RATE'`, `'YIELD_BID'`, `'YIELD_ASK'`, `'YIELD_BID_ASK'`, `'YIELD_LAST'`.  
    - For `'SCHEDULE'`, use `:meth:.reqHistoricalSchedule`.  
- **useRTH**:  
    - If `True`, only show data from within Regular Trading Hours.  
    - If `False`, show all data.  
- **formatDate**:  
    - For an intraday request, setting to `2` will cause the returned date fields to be timezone-aware `datetime.datetime` with UTC timezone, instead of local timezone as used by TWS.  
- **keepUpToDate**:  
    - If `True`, a realtime subscription is started to keep the bars updated.  
    - `endDateTime` must be set empty (`''`) then.  
- **chartOptions**: Unknown.  
- **timeout**:  
    - Timeout in seconds after which to cancel the request and return an empty bar series. 
    - If the data request is huge, this parameter could spoil the request  
    - Set to `0` to wait indefinitely.  

In [ ]:
historical_data_interval = '10 secs' # Candle period to fetch
request_duration = '6 Y'  # Duration in days (use D, not "day"). Use a very big value if you want the maximum historical data, it will fetch the maximum available automatically.
price_source = 'TRADES'  # 'BID', 'ASK', or 'TRADES' (note that for some symbols, (e.g. EURUSD) only 'BID' and 'ASK' are available)

bars = ib.reqHistoricalData(
        contract,
        endDateTime=end_date,
        durationStr=str(request_duration),
        barSizeSetting=str(historical_data_interval),
        whatToShow=price_source,
        useRTH=False, 
        formatDate=2,
        timeout = 0)

bars[0]
new_df = util.df(bars)

display(new_df.head())
display(new_df.tail())
# Remove the 'volume', 'average', and 'barCount' columns from the DataFrame
new_df = new_df.drop(columns=['volume', 'average', 'barCount'])

# Display the updated DataFrame
new_df.head()

# Nouvelle cellule pour le premier téléchargement de données (pas de fichier existant)

import os

# save_path doit être défini comme dans la cellule précédente
# save_path = f"../marketData/{contract.symbol}_10secs_20240915_to_20250826_{price_source}.csv"

save_path = f"../marketData/{contract.symbol}_10secs_20240915_to_20250826_{price_source}.csv" # Here, enter the correct file path for the new csv data file

# Sauvegarder le DataFrame nouvellement téléchargé
new_df.to_csv(save_path, index=True)
print(f"New data saved to: {save_path}")

# Vérification optionnelle
import pandas as pd
df_check = pd.read_csv(save_path, index_col=0)
display(df_check.head())
display(df_check.tail())

In [16]:
# Cellule pour télécharger une période spécifique
import pandas as pd
from datetime import datetime

# Définir la période à télécharger
end_date = '20220823 23:59:59'  # Format: YYYYMMDD HH:MM:SS
segment_duration = '10 Y'        # Période de 6 mois
price_source = 'TRADES'         # ou 'BID', 'ASK', etc.
contract = Index('SPX', 'CBOE', 'USD')  # À adapter selon le contrat souhaité

# Télécharger les données pour cette période
bars = ib.reqHistoricalData(
    contract,
    endDateTime=end_date,
    durationStr=segment_duration,
    barSizeSetting='1 min',
    whatToShow=price_source,
    useRTH=False,
    formatDate=2,
    timeout=0
)

# Convertir en dataframe
period_df = util.df(bars)
display(period_df.head())
display(period_df.tail())
period_df = period_df.drop(columns=['volume', 'average', 'barCount'])

# Afficher un aperçu
print(f"Données téléchargées: {len(period_df)} barres")
print(f"Première date: {period_df['date'].min()}")
print(f"Dernière date: {period_df['date'].max()}")

# Sauvegarder cette période dans un fichier CSV
start_date_str = period_df['date'].min().strftime('%Y%m%d')
end_date_str = period_df['date'].max().strftime('%Y%m%d')
file_name = f"{contract.symbol}_1min_{start_date_str}_to_{end_date_str}_{price_source}.csv"
save_path = f"../marketData/{file_name}"

period_df.to_csv(save_path, index=True)
print(f"Données sauvegardées dans: {save_path}")

KeyboardInterrupt: 

In [15]:
# Cellule pour fusionner plusieurs fichiers CSV
import pandas as pd
import os
from Helpers import merge_ohlc_dataframes

# Liste des fichiers à fusionner
file_paths = [
    "../marketData/SPX_10secs_20220822_to_20230303_TRADES.csv",
    "../marketData/SPX_10secs_20230302_to_20250826_TRADES.csv",
    # Ajoutez d'autres fichiers selon besoin
]

# Charger le premier fichier
combined_df = pd.read_csv(file_paths[0], index_col=0)
print(f"Chargé {file_paths[0]}: {len(combined_df)} lignes")

# Fusionner les fichiers suivants
for file_path in file_paths[1:]:
    if os.path.exists(file_path):
        next_df = pd.read_csv(file_path, index_col=0)
        print(f"Chargé {file_path}: {len(next_df)} lignes")
        
        # Fusionner avec le dataframe combiné
        combined_df = merge_ohlc_dataframes(combined_df, next_df, frequency='10s')
        print(f"Après fusion: {len(combined_df)} lignes")
    else:
        print(f"Fichier non trouvé: {file_path}")

# Vérifier les résultats
print("\nDataframe final:")
print(f"Nombre de lignes: {len(combined_df)}")
print(f"Période: de {combined_df['date'].min()} à {combined_df['date'].max()}")

# Sauvegarder le résultat final
start_date_str = combined_df['date'].min().strftime('%Y%m%d')
end_date_str = combined_df['date'].max().strftime('%Y%m%d')
final_path = f"../marketData/{contract.symbol}_10secs_{start_date_str}_to_{end_date_str}_{price_source}.csv"
combined_df.to_csv(final_path, index=True)
print(f"Dataframe fusionné sauvegardé dans: {final_path}")

Chargé ../marketData/SPX_10secs_20220822_to_20230303_TRADES.csv: 312480 lignes
Chargé ../marketData/SPX_10secs_20230302_to_20250826_TRADES.csv: 1453680 lignes
First few missing timestamps: [Timestamp('2022-08-22 20:00:00+0000', tz='UTC'), Timestamp('2022-08-22 20:00:10+0000', tz='UTC'), Timestamp('2022-08-22 20:00:20+0000', tz='UTC'), Timestamp('2022-08-22 20:00:30+0000', tz='UTC'), Timestamp('2022-08-22 20:00:40+0000', tz='UTC')]
Après fusion: 1761480 lignes

Dataframe final:
Nombre de lignes: 1761480
Période: de 2022-08-22 13:30:00+00:00 à 2025-08-26 19:59:50+00:00
Dataframe fusionné sauvegardé dans: ../marketData/SPX_10secs_20220822_to_20250826_TRADES.csv


#### 5. Convert the list of bars to a data frame, print the first / last rows and remove useless columns:

In [ ]:
bars[0]
new_df = util.df(bars)

display(new_df.head())
display(new_df.tail())
# Remove the 'volume', 'average', and 'barCount' columns from the DataFrame
new_df = new_df.drop(columns=['volume', 'average', 'barCount'])

# Display the updated DataFrame
new_df.head()

Now, you have a **pandas DataFrame** containing the historical data for your chosen contract. You can use this DataFrame for further analysis or visualization as needed. 
In the next cell, you can update an existing csv file by merging it with the new data. 

## Additional features

#### DataFrame update (only for existing data)
Update the dataframe by merging the new data with old ones

**IMPORTANT**: Use the correct format file : `../marketData/{contract.symbol}_{candle_period}_{first_date}_to_{last_date}_{price_source}.csv`

Date format to use: `YYYYMMDD`

In [ ]:
import pandas as pd
from Helpers import merge_ohlc_dataframes
import os

# Load your existing data - use index_col=0 to treat first column as index
existing_file_path = "../marketData/NDX_10secs_20220214_to_20250716_TRADES.csv" # Here, enter the correct file path fo the existing csv data file
existing_df = pd.read_csv(existing_file_path, index_col=0)
display(existing_df.head())

# Merge the dataframes
merged_df = merge_ohlc_dataframes(existing_df, new_df, frequency='10s') # Adjust frequency as needed
display(merged_df.head())
display(merged_df.tail())

# Save the merged dataframe
save_path = f"../marketData/{contract.symbol}_10secs_20240915_to_20250826_{price_source}.csv" # Here, enter the correct file path for the new csv data file
merged_df.to_csv(save_path, index=True)
print(f"Merged data saved to: {save_path}")
# Delete the original file if needed
if os.path.exists(existing_file_path):
    os.remove(existing_file_path)
    print(f"Deleted original file: {existing_file_path}")


In [ ]:
# Nouvelle cellule pour le premier téléchargement de données (pas de fichier existant)

import os

# save_path doit être défini comme dans la cellule précédente
# save_path = f"../marketData/{contract.symbol}_10secs_20240915_to_20250826_{price_source}.csv"

save_path = f"../marketData/{contract.symbol}_10secs_20240915_to_20250826_{price_source}.csv" # Here, enter the correct file path for the new csv data file

# Sauvegarder le DataFrame nouvellement téléchargé
new_df.to_csv(save_path, index=True)
print(f"New data saved to: {save_path}")

# Vérification optionnelle
import pandas as pd
df_check = pd.read_csv(save_path, index_col=0)
display(df_check.head())
display(df_check.tail())

#### Checking data integrity

In [ ]:
from Helpers import checkDataFile, visualize_data_gaps
import matplotlib.pyplot as plt

symbol = 'EUR'
interval = '10secs'
start_date = '20240915'
end_date = '20250717'
price_source= 'ASK'

# 1. Load the existing dataframe
save_path = f"../marketData/{symbol}_{interval}_{start_date}_to_{end_date}_{price_source}.csv"

# Analyze data gaps
report = checkDataFile(
    file_path=save_path, 
    interval=interval
)

# Print summary
print(f"Analyzed {report['total_trading_days']} trading days")
print(f"Found {report['days_with_gaps']} days with gaps ({report['analysis_summary']['gap_percentage']:.2f}%)")
print(f"Total gaps detected: {report['total_gaps']}")

# Visualize the gaps
fig = visualize_data_gaps(report)
plt.show()

# To examine specific days with large gaps
problem_days = {date: data for date, data in report["gaps_by_date"].items() 
                if data["missing_points"] > 10}
print(f"Days with more than 10 missing points: {len(problem_days)}")
for date, data in sorted(problem_days.items()):
    print(f"{date}: Missing {data['missing_points']} of {data['expected_points']} points")